In [1]:
# ============================================
# Startup Cell: Mount Drive + Verify Metadata Inputs
# ============================================

from google.colab import drive
drive.mount("/content/drive")

import os
import sys

BASE_DIR = "/content/drive/MyDrive/DIP_Project"
SRC_DIR = os.path.join(BASE_DIR, "src")

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

from project_config import (
    METADATA_DIR,
    DATASET_CODES,
    PREPROCESSED_METADATA_COLUMNS,
    AI_LABEL,
    REAL_LABEL,
    TRAIN_RATIO,
    TEST_RATIO,
    TRAIN_SUBSET,
    TEST_SUBSET,
    TRAIN_PER_SOURCE,
    TEST_PER_SOURCE,
    RANDOM_SEED,
    COMBINED_METADATA_FILENAME,
    TRAIN_METADATA_FILENAME,
    TEST_METADATA_FILENAME,
)

# -------------------------------------------------
# Input metadata CSVs (from preprocessing stage)
# -------------------------------------------------
INPUT_FILES = [
    "imgn_preprocessed_metadata.csv",
    "coco_preprocessed_metadata.csv",
    "diff_preprocessed_metadata.csv",
    "sdxl_preprocessed_metadata.csv",
    "mj_preprocessed_metadata.csv",
    "open_preprocessed_metadata.csv"
]

# -------------------------------------------------
# Verification
# -------------------------------------------------
print("Verifying required metadata CSV files in METADATA_DIR...\n")

all_ok = True

for filename in INPUT_FILES:
    path = os.path.join(METADATA_DIR, filename)
    exists = os.path.exists(path)
    print(f"{filename:<50} {'OK' if exists else 'MISSING'}")
    all_ok = all_ok and exists

if not all_ok:
    raise FileNotFoundError("One or more required metadata CSV files are missing from METADATA_DIR.")

print("\nAll required metadata CSV files are present.")
print(f"Metadata directory: {METADATA_DIR}")


Mounted at /content/drive
Verifying required metadata CSV files in METADATA_DIR...

imgn_preprocessed_metadata.csv                     OK
coco_preprocessed_metadata.csv                     OK
diff_preprocessed_metadata.csv                     OK
sdxl_preprocessed_metadata.csv                     OK
mj_preprocessed_metadata.csv                       OK
open_preprocessed_metadata.csv                     OK

All required metadata CSV files are present.
Metadata directory: /content/drive/MyDrive/DIP_Project/data/metadata


In [2]:
# ============================================
# Cell 0: Notebook Overview
# ============================================
# Purpose:
#   This notebook combines 6 preprocessed dataset metadata CSV files
#   and creates balanced train / test split CSV files.
#
# Inputs:
#   The notebook expects these 6 preprocessed metadata CSV files
#   in the project metadata folder:
#     /content/drive/MyDrive/DIP_Project/data/metadata/imgn_preprocessed_metadata.csv
#     /content/drive/MyDrive/DIP_Project/data/metadata/coco_preprocessed_metadata.csv
#     /content/drive/MyDrive/DIP_Project/data/metadata/diff_preprocessed_metadata.csv
#     /content/drive/MyDrive/DIP_Project/data/metadata/sdxl_preprocessed_metadata.csv
#     /content/drive/MyDrive/DIP_Project/data/metadata/mj_preprocessed_metadata.csv
#     /content/drive/MyDrive/DIP_Project/data/metadata/open_preprocessed_metadata.csv
#
# Assumptions:
#   - Each input CSV contains a 'filename' column.
#   - Each input CSV contains 3000 rows.
#   - The datasets correspond to:
#       ImageNet_1K_256       -> real
#       MS_COCO_2017          -> real
#       OpenImages            -> real
#       DiffusionDB           -> ai
#       SDXL_Generated_10K    -> ai
#       Midjourney            -> ai
#   - This notebook does NOT move, copy, unzip, or preprocess images.
#   - This notebook reads metadata CSV files from Google Drive and
#     writes output metadata CSV files back to Google Drive.
#
# What the notebook does:
#   Cell 1:
#     Define paths using the project_config.py file.
#
#   Cell 2:
#     Define dataset configuration, including CSV filenames and class labels.
#
#   Cell 3:
#     Load the 6 input CSV files, add:
#       - source_dataset
#       - class_label
#       - image_path
#     Then combine them into one master dataframe and save:
#       combined_metadata.csv
#
#   Cell 4:
#     For each dataset independently:
#       1. Shuffle rows
#       2. Split into exact counts:
#            train = 2400
#            test  =  600
#     Then:
#       - combine all train splits
#       - combine all test splits
#       - shuffle each final subset independently
#     Then save:
#       train_metadata.csv
#       test_metadata.csv
#
#   Cell 5:
#     Perform an optional validation that the output CSV files exist
#     in the metadata folder and report their shapes.
#
# Outputs:
#   /content/drive/MyDrive/DIP_Project/data/metadata/combined_metadata.csv
#   /content/drive/MyDrive/DIP_Project/data/metadata/train_metadata.csv
#   /content/drive/MyDrive/DIP_Project/data/metadata/test_metadata.csv
#
# Expected final sizes:
#   combined_metadata.csv     -> 18000 rows
#   train_metadata.csv        -> 14400 rows
#   test_metadata.csv         ->  3600 rows
#
# Notes:
#   - The split is balanced by source dataset and by class label.
#   - The split is reproducible because a fixed random seed is used.
#   - The training set will later be used for k-fold cross-validation.
#   - Images remain in their existing storage location; the CSV files
#     act as the split definitions for later notebooks.
#   - Persistent metadata artifacts are stored in the project
#     metadata folder on Google Drive, not in temporary local runtime storage.
# ============================================

print("Notebook overview loaded.")


Notebook overview loaded.


In [3]:
# =========================
# Cell 1: Define paths
# =========================

from pathlib import Path
import pandas as pd

INPUT_CSV_DIR = Path(METADATA_DIR)
OUTPUT_DIR = Path(METADATA_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Paths defined.")


Paths defined.


In [4]:
# =========================
# Cell 2: Dataset configuration
# =========================

DATASET_CONFIG = {
    "ImageNet_1K_256": {
        "csv": "imgn_preprocessed_metadata.csv",
        "class_label": REAL_LABEL,
    },
    "MS_COCO_2017": {
        "csv": "coco_preprocessed_metadata.csv",
        "class_label": REAL_LABEL,
    },
    "OpenImages": {
        "csv": "open_preprocessed_metadata.csv",
        "class_label": REAL_LABEL,
    },
    "DiffusionDB": {
        "csv": "diff_preprocessed_metadata.csv",
        "class_label": AI_LABEL,
    },
    "SDXL_Generated_10K": {
        "csv": "sdxl_preprocessed_metadata.csv",
        "class_label": AI_LABEL,
    },
    "Midjourney": {
        "csv": "mj_preprocessed_metadata.csv",
        "class_label": AI_LABEL,
    },
}

print("Dataset configurations defined.")


Dataset configurations defined.


In [5]:
# =========================
# Cell 3: Load and combine metadata
# =========================
dfs = []

BASE_COLUMNS = ["filename"]

for dataset_name, cfg in DATASET_CONFIG.items():
    csv_path = INPUT_CSV_DIR / cfg["csv"]

    if not csv_path.exists():
        raise FileNotFoundError(f"Missing input CSV: {csv_path}")

    df = pd.read_csv(csv_path)

    if "filename" not in df.columns:
        raise ValueError(f"'filename' column not found in {csv_path}")

    # Keep only required columns
    df = df[BASE_COLUMNS].copy()

    # Add standardized fields
    df["source_dataset"] = dataset_name
    df["class_label"] = cfg["class_label"]

    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)

print("Combined dataset shape:", combined_df.shape)

print("\nCounts by source_dataset:")
print(combined_df["source_dataset"].value_counts())

print("\nCounts by class_label:")
print(combined_df["class_label"].value_counts())

print("\nColumns in combined dataset:")
print(combined_df.columns.tolist())

combined_csv_path = OUTPUT_DIR / COMBINED_METADATA_FILENAME
combined_df.to_csv(combined_csv_path, index=False)

print(f"\nSaved combined metadata to: {combined_csv_path}")


Combined dataset shape: (18000, 3)

Counts by source_dataset:
source_dataset
ImageNet_1K_256       3000
MS_COCO_2017          3000
OpenImages            3000
DiffusionDB           3000
SDXL_Generated_10K    3000
Midjourney            3000
Name: count, dtype: int64

Counts by class_label:
class_label
rl    9000
ai    9000
Name: count, dtype: int64

Columns in combined dataset:
['filename', 'source_dataset', 'class_label']

Saved combined metadata to: /content/drive/MyDrive/DIP_Project/data/metadata/combined_metadata.csv


In [6]:
# =========================
# Cell 4: Split combined metadata into exact train / test sets
# =========================

split_dfs = []

for dataset_name, group_df in combined_df.groupby("source_dataset"):
    # Shuffle each dataset independently
    group_df = group_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

    # Exact-count split
    train_df = group_df.iloc[:TRAIN_PER_SOURCE].copy()
    test_df = group_df.iloc[
        TRAIN_PER_SOURCE:TRAIN_PER_SOURCE + TEST_PER_SOURCE
    ].copy()

    train_df["subset"] = TRAIN_SUBSET
    test_df["subset"] = TEST_SUBSET

    split_dfs.extend([train_df, test_df])

split_df = pd.concat(split_dfs, ignore_index=True)

train_metadata = split_df[split_df["subset"] == TRAIN_SUBSET].copy()
test_metadata = split_df[split_df["subset"] == TEST_SUBSET].copy()

# Final shuffle of each subset independently
train_metadata = train_metadata.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
test_metadata = test_metadata.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

train_csv_path = OUTPUT_DIR / TRAIN_METADATA_FILENAME
test_csv_path = OUTPUT_DIR / TEST_METADATA_FILENAME

train_metadata.to_csv(train_csv_path, index=False)
test_metadata.to_csv(test_csv_path, index=False)

print("Train shape:", train_metadata.shape)
print("Test shape:", test_metadata.shape)

print("\nCounts by subset:")
print(pd.Series({
    TRAIN_SUBSET: len(train_metadata),
    TEST_SUBSET: len(test_metadata),
}))

print("\nCounts by subset and source_dataset:")
print(split_df.groupby(["subset", "source_dataset"]).size())

print("\nCounts by subset and class_label:")
print(split_df.groupby(["subset", "class_label"]).size())

print(f"\nSaved train metadata to: {train_csv_path}")
print(f"Saved test metadata to: {test_csv_path}")


Train shape: (14400, 4)
Test shape: (3600, 4)

Counts by subset:
train    14400
test      3600
dtype: int64

Counts by subset and source_dataset:
subset  source_dataset    
test    DiffusionDB            600
        ImageNet_1K_256        600
        MS_COCO_2017           600
        Midjourney             600
        OpenImages             600
        SDXL_Generated_10K     600
train   DiffusionDB           2400
        ImageNet_1K_256       2400
        MS_COCO_2017          2400
        Midjourney            2400
        OpenImages            2400
        SDXL_Generated_10K    2400
dtype: int64

Counts by subset and class_label:
subset  class_label
test    ai             1800
        rl             1800
train   ai             7200
        rl             7200
dtype: int64

Saved train metadata to: /content/drive/MyDrive/DIP_Project/data/metadata/train_metadata.csv
Saved test metadata to: /content/drive/MyDrive/DIP_Project/data/metadata/test_metadata.csv


In [7]:
# =========================
# Cell 5: Optional validation of output CSV files
# =========================
for csv_name in [
    COMBINED_METADATA_FILENAME,
    TRAIN_METADATA_FILENAME,
    TEST_METADATA_FILENAME,
]:
    csv_path = OUTPUT_DIR / csv_name
    if not csv_path.exists():
        print(f"Missing: {csv_path}")
    else:
        df = pd.read_csv(csv_path)
        print(f"{csv_name}: {df.shape}")


combined_metadata.csv: (18000, 3)
train_metadata.csv: (14400, 4)
test_metadata.csv: (3600, 4)
